# Actividad 5: Analítica de órdenes de producción

**Autores:** Laila Tatiana Cardenas Guerrero, Luis Eduardo Ortega Montes  
**Unidad de análisis:** orden de producción  
**Usuarios:** Líder de Producción y Analista de Costos

**Pregunta:** ¿Qué órdenes deben reportarse por no cumplir los criterios de cierre y liberación?

El flujo se ejecuta por etapas. Valida cada celda antes de continuar.

# contexto y diagrama

## 1. Librerías

In [40]:
import re
import unicodedata
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown

from html import escape
from IPython.display import display, HTML
from pathlib import Path
import shutil

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 2. Rutas y parámetros

In [41]:
URL_BASE = "https://raw.githubusercontent.com/luisort1/Actividad_5_GR_4_Produci-n/main/"
ARCHIVOS = {
    "cabeceras": "Cabeceras.csv",
    "consumos": "Consumo.csv",
    "productos": "Productos.csv",
    "recetas": "RECETAS.csv",
    "conversiones": "CONVERSIONES.csv",
    "tolerancias_peso": "TOLERANCIAS_PESO.csv",
    "tolerancias_consumo": "TOLERANCIAS_CONSUMO.csv",
    "reglas_especiales": "REGLAS_ESPECIALES.csv",
}
CODIFICACIONES = {nombre: "utf-8-sig" for nombre in ARCHIVOS}
CODIFICACIONES["cabeceras"] = "latin1"
MOVIMIENTOS_PRODUCCION = {"101", "102"}
MOVIMIENTOS_CONSUMO = {"261", "262"}
MOVIMIENTOS_REVISION = {"531", "532"}
FECHA_EVALUACION = pd.Timestamp.now().normalize()
RUTA_SALIDA = Path("resultados_actividad_5")
# RUTA_SALIDA.mkdir(exist_ok=True)

## 3. Carga original de las ocho fuentes

In [42]:
def cargar_fuentes(url_base: str, archivos: Dict[str, str], codificaciones: Dict[str, str]) -> Dict[str, pd.DataFrame]:
    """Carga los CSV sin normalizar valores ni encabezados."""
    tablas, errores = {}, []
    for nombre, archivo in archivos.items():
        try:
            tablas[nombre] = pd.read_csv(
                url_base + archivo,
                sep=";",
                encoding=codificaciones[nombre],
                dtype="string",
            )
        except Exception as exc:
            errores.append({"tabla": nombre, "archivo": archivo, "error": str(exc)})
    if errores:
        display(pd.DataFrame(errores))
        raise RuntimeError("No fue posible cargar todas las fuentes.")
    return tablas

ORIGINALES = cargar_fuentes(URL_BASE, ARCHIVOS, CODIFICACIONES)
print("Fuentes cargadas:", len(ORIGINALES))

Fuentes cargadas: 8


## 4. Vista de tablas originales

In [43]:
DIMENSIONES_ORIGINALES = pd.DataFrame([
    {"tabla": nombre, "filas": tabla.shape[0], "columnas": tabla.shape[1], "estado": "OK" if len(tabla) and len(tabla.columns) > 1 else "ERROR"}
    for nombre, tabla in ORIGINALES.items()
])
display(DIMENSIONES_ORIGINALES)

for nombre, tabla in ORIGINALES.items():
    print(f"\n{nombre}: {tabla.shape[0]} filas x {tabla.shape[1]} columnas")
    print("Encabezados originales:", list(tabla.columns))
    display(tabla.head(3))

if DIMENSIONES_ORIGINALES["estado"].eq("ERROR").any():
    raise ValueError("Una fuente está vacía o no se separó correctamente.")

,tabla,filas,columnas,estado
0,cabeceras,149,19,OK
1,consumos,5341,17,OK
2,productos,6,10,OK
3,recetas,25,11,OK
4,conversiones,28,10,OK
5,tolerancias_peso,6,10,OK
6,tolerancias_consumo,25,13,OK
7,reglas_especiales,2,8,OK



cabeceras: 149 filas x 19 columnas
Encabezados originales: ['Centro planificación', 'Orden', 'Almacén', 'Material', 'Lote', 'Texto breve material', 'Cantidad orden', 'Cantidad entregada', 'Fecha de fin real', 'Unidad de medida', 'Estado de sistema', 'Status de usuario', 'Hora creación', 'Fecha inicio extrema', 'Fecha fin extrema', 'Autor', 'Planificador nec.', 'Versión fabricación', 'Modificado por']


,Centro planificación,Orden,Almacén,Material,Lote,Texto breve material,Cantidad orden,Cantidad entregada,Fecha de fin real,Unidad de medida,Estado de sistema,Status de usuario,Hora creación,Fecha inicio extrema,Fecha fin extrema,Autor,Planificador nec.,Versión fabricación,Modificado por
0,B010,1082938,1003,12000496,2605180823,MIX ADEREZO ENSALADA,948,948,18/5/2026,KG,CERR NOTP ENTR PREC EDET DDPN DMNV LINA*,RECT,11:02:46 a. m.,18/5/2026,19/5/2026,USUARIO 1,P09,V001,CABAD
1,B010,1083068,1003,12000496,2605210584,MIX ADEREZO ENSALADA,948,948,21/5/2026,KG,CERR NOTP ENTR PREC EDET DDPN DMNV LINA*,RECT,9:01:25 a. m.,21/5/2026,22/5/2026,USUARIO 2,P09,V001,CABAD
2,B010,1083162,1003,12000496,2605230326,MIX ADEREZO ENSALADA,474,474,23/5/2026,KG,CERR NOTP ENTR PREC EDET DDPN DMNV LINA*,LIB RECT,8:07:18 a. m.,23/5/2026,23/5/2026,USUARIO 1,P09,V001,CABAD



consumos: 5341 filas x 17 columnas
Encabezados originales: ['Orden', 'Material', 'Documento material', 'Descripción material', 'Ctd.en UM base y signo +/-', 'Impte.mon.local', 'Clase de movimiento', 'Fecha de documento', 'Fecha contabiliz.', 'Almacén', 'Ind.movim.mercancías', 'Centro', 'Pos.documento mat.', 'Lote', 'Indicador Debe/Haber', 'Moneda', 'Reserva']


,Orden,Material,Documento material,Descripción material,Ctd.en UM base y signo +/-,Impte.mon.local,Clase de movimiento,Fecha de documento,Fecha contabiliz.,Almacén,Ind.movim.mercancías,Centro,Pos.documento mat.,Lote,Indicador Debe/Haber,Moneda,Reserva
0,1082908,11001135,4916453391,MARINADO PICANTE CJX25UN,55,1.569.209,261,19/5/2026,19/5/2026,1001,1,B010,1,2604221405,H,COP,1946599
1,1082908,11001135,4916459788,MARINADO PICANTE CJX25UN,55,1.569.209,261,19/5/2026,19/5/2026,1001,1,B010,1,2605070456,H,COP,1946599
2,1082908,12002532,4916438027,ALAS CORTADAS POLLO,"764,700",8.124.173,261,18/5/2026,18/5/2026,1001,1,B010,1,2605140096,H,COP,1946599


...

,cod_producto,nivel_control,unidad_control,peso_minimo_unitario_kg,peso_maximo_unitario_kg,peso_minimo_presentacion_kg,peso_maximo_presentacion_kg,tolerancia_porcentaje,genera_novedad,estado_parametrizacion
0,14002692,UNITARIO,KG_UNIDAD,"0,18","0,195",<NA>,<NA>,<NA>,SI,PARAMETRIZADO
1,14003025,UNITARIO,KG_UNIDAD,"0,12","0,13",<NA>,<NA>,<NA>,SI,PARAMETRIZADO
2,12002518,PRESENTACION,KG,<NA>,<NA>,2,<NA>,<NA>,SI,PARAMETRIZADO



tolerancias_consumo: 25 filas x 13 columnas
Encabezados originales: ['cod_producto', 'cod_componente', 'tipo_componente', 'metodo_validacion', 'tolerancia_inferior', 'tolerancia_superior', 'unidad_tolerancia', 'validar_individualmente', 'permite_merma', 'merma_maxima_porcentaje', 'genera_novedad', 'estado_parametrizacion', 'observacion']


,cod_producto,cod_componente,tipo_componente,metodo_validacion,tolerancia_inferior,tolerancia_superior,unidad_tolerancia,validar_individualmente,permite_merma,merma_maxima_porcentaje,genera_novedad,estado_parametrizacion,observacion
0,14002692,11001128,MP,PORCENTAJE,4,4,PORCENTAJE,SI,NO,<NA>,SI,PARAMETRIZADO,<NA>
1,14002692,12002544,MP,PORCENTAJE,3,3,PORCENTAJE,SI,NO,<NA>,SI,PARAMETRIZADO,<NA>
2,14002692,11001132,MP,PORCENTAJE,3,3,PORCENTAJE,SI,NO,<NA>,SI,PARAMETRIZADO,<NA>



reglas_especiales: 2 filas x 8 columnas
Encabezados originales: ['cod_producto', 'tipo_regla', 'valor_regla', 'unidad_regla', 'fecha_base', 'resultado_si_no_cumple', 'estado_parametrizacion', 'observacion']


,cod_producto,tipo_regla,valor_regla,unidad_regla,fecha_base,resultado_si_no_cumple,estado_parametrizacion,observacion
0,14002692,TEMPERATURA,2,DIAS_CALENDARIO,fecha_de_fin_real,Esperar control de temperatura,PARAMETRIZADO,<NA>
1,14003025,TEMPERATURA,2,DIAS_CALENDARIO,fecha_de_fin_real,Esperar control de temperatura,PARAMETRIZADO,<NA>


## 5. Funciones de normalización

In [44]:
def normalizar_nombre(nombre: str) -> str:
    """Deja el encabezado en formato snake_case."""
    texto = unicodedata.normalize("NFKD", str(nombre)).encode("ascii", "ignore").decode().lower().strip()
    texto = texto.replace("+/-", "signo").replace("+", "mas")
    return re.sub(r"[^a-z0-9]+", "_", texto).strip("_")


def convertir_numero_es(serie: pd.Series) -> pd.Series:
    """Convierte punto de miles y coma decimal."""
    texto = (serie.astype("string").str.strip()
             .str.replace(".", "", regex=False)
             .str.replace(",", ".", regex=False))
    return pd.to_numeric(texto, errors="coerce")


def preparar_fuentes(originales: Dict[str, pd.DataFrame]) -> Dict[str, pd.DataFrame]:
    """Normaliza copias de las fuentes y conserva los originales."""
    tablas = {}
    for nombre, original in originales.items():
        tabla = original.copy()
        tabla.columns = [normalizar_nombre(c) for c in tabla.columns]
        tabla = tabla.loc[:, ~tabla.columns.str.match(r"^unnamed|^$")].copy()
        for columna in tabla.select_dtypes(include="string").columns:
            tabla[columna] = tabla[columna].str.strip()
        tablas[nombre] = tabla

    cantidad = [c for c in tablas["consumos"].columns if c.startswith("ctd_en_um_base")]
    if len(cantidad) != 1:
        raise ValueError(f"Cantidad SAP ambigua: {cantidad}")
    tablas["consumos"] = tablas["consumos"].rename(columns={cantidad[0]: "cantidad_movimiento"})

    for tabla in tablas.values():
        for columna in ["orden", "material", "cod_producto", "cod_componente", "lote", "documento_material"]:
            if columna in tabla.columns:
                tabla[columna] = tabla[columna].astype("string").str.replace(r"\.0$", "", regex=True)

    numericas = {
        "cabeceras": ["cantidad_orden", "cantidad_entregada"],
        "consumos": ["cantidad_movimiento"],
        "productos": ["unidades_por_presentacion", "peso_referencia_unitario_kg", "peso_referencia_presentacion_kg", "dias_temperatura"],
        "recetas": ["cantidad_receta", "peso_estandar_kg"],
        "conversiones": ["factor_a_unidad_destino", "factor_consumo_real_a_control"],
        "tolerancias_peso": ["peso_minimo_unitario_kg", "peso_maximo_unitario_kg", "peso_minimo_presentacion_kg", "peso_maximo_presentacion_kg", "tolerancia_porcentaje"],
        "tolerancias_consumo": ["tolerancia_inferior", "tolerancia_superior", "merma_maxima_porcentaje"],
        "reglas_especiales": ["valor_regla"],
    }
    for nombre, columnas in numericas.items():
        for columna in columnas:
            if columna in tablas[nombre].columns:
                tablas[nombre][columna] = convertir_numero_es(tablas[nombre][columna])

    for columna in ["fecha_inicio_extrema", "fecha_de_fin_real"]:
        if columna in tablas["cabeceras"].columns:
            tablas["cabeceras"][columna] = pd.to_datetime(tablas["cabeceras"][columna], dayfirst=True, errors="coerce")
    for columna in ["fecha_de_documento", "fecha_contabiliz"]:
        if columna in tablas["consumos"].columns:
            tablas["consumos"][columna] = pd.to_datetime(tablas["consumos"][columna], dayfirst=True, errors="coerce")
    return tablas

## 6. Normalización y dimensiones posteriores

In [45]:
TABLAS = preparar_fuentes(ORIGINALES)
DIMENSIONES_NORMALIZADAS = pd.DataFrame([
    {"tabla": nombre, "filas": tabla.shape[0], "columnas": tabla.shape[1],
     "nulos": int(tabla.isna().sum().sum()), "duplicados_exactos": int(tabla.duplicated().sum())}
    for nombre, tabla in TABLAS.items()
])
display(DIMENSIONES_NORMALIZADAS)

for nombre, tabla in TABLAS.items():
    print(f"\n{nombre}: {tabla.shape[0]} filas x {tabla.shape[1]} columnas")
    print("Columnas normalizadas:", list(tabla.columns))
    display(tabla.head(3))

,tabla,filas,columnas,nulos,duplicados_exactos
0,cabeceras,149,19,0,0
1,consumos,5341,17,0,0
2,productos,6,10,0,0
3,recetas,25,11,25,0
4,conversiones,28,10,0,0
5,tolerancias_peso,6,10,20,0
6,tolerancias_consumo,25,13,50,0
7,reglas_especiales,2,8,2,0



cabeceras: 149 filas x 19 columnas
Columnas normalizadas: ['centro_planificacion', 'orden', 'almacen', 'material', 'lote', 'texto_breve_material', 'cantidad_orden', 'cantidad_entregada', 'fecha_de_fin_real', 'unidad_de_medida', 'estado_de_sistema', 'status_de_usuario', 'hora_creacion', 'fecha_inicio_extrema', 'fecha_fin_extrema', 'autor', 'planificador_nec', 'version_fabricacion', 'modificado_por']


,centro_planificacion,orden,almacen,material,lote,texto_breve_material,cantidad_orden,cantidad_entregada,fecha_de_fin_real,unidad_de_medida,estado_de_sistema,status_de_usuario,hora_creacion,fecha_inicio_extrema,fecha_fin_extrema,autor,planificador_nec,version_fabricacion,modificado_por
0,B010,1082938,1003,12000496,2605180823,MIX ADEREZO ENSALADA,948.0000,948.0000,2026-05-18,KG,CERR NOTP ENTR PREC EDET DDPN DMNV LINA*,RECT,11:02:46 a. m.,2026-05-18,19/5/2026,USUARIO 1,P09,V001,CABAD
1,B010,1083068,1003,12000496,2605210584,MIX ADEREZO ENSALADA,948.0000,948.0000,2026-05-21,KG,CERR NOTP ENTR PREC EDET DDPN DMNV LINA*,RECT,9:01:25 a. m.,2026-05-21,22/5/2026,USUARIO 2,P09,V001,CABAD
2,B010,1083162,1003,12000496,2605230326,MIX ADEREZO ENSALADA,474.0000,474.0000,2026-05-23,KG,CERR NOTP ENTR PREC EDET DDPN DMNV LINA*,LIB RECT,8:07:18 a. m.,2026-05-23,23/5/2026,USUARIO 1,P09,V001,CABAD



consumos: 5341 filas x 17 columnas
Columnas normalizadas: ['orden', 'material', 'documento_material', 'descripcion_material', 'cantidad_movimiento', 'impte_mon_local', 'clase_de_movimiento', 'fecha_de_documento', 'fecha_contabiliz', 'almacen', 'ind_movim_mercancias', 'centro', 'pos_documento_mat', 'lote', 'indicador_debe_haber', 'moneda', 'reserva']


,orden,material,documento_material,descripcion_material,cantidad_movimiento,impte_mon_local,clase_de_movimiento,fecha_de_documento,fecha_contabiliz,almacen,ind_movim_mercancias,centro,pos_documento_mat,lote,indicador_debe_haber,moneda,reserva
0,1082908,11001135,4916453391,MARINADO PICANTE CJX25UN,55.0000,1.569.209,261,2026-05-19,2026-05-19,1001,1,B010,1,2604221405,H,COP,1946599
1,1082908,11001135,4916459788,MARINADO PICANTE CJX25UN,55.0000,1.569.209,261,2026-05-19,2026-05-19,1001,1,B010,1,2605070456,H,COP,1946599
2,1082908,12002532,4916438027,ALAS CORTADAS POLLO,764.7000,8.124.173,261,2026-05-18,2026-05-18,1001,1,B010,1,2605140096,H,COP,1946599


...

,cod_producto,nivel_control,unidad_control,peso_minimo_unitario_kg,peso_maximo_unitario_kg,peso_minimo_presentacion_kg,peso_maximo_presentacion_kg,tolerancia_porcentaje,genera_novedad,estado_parametrizacion
0,14002692,UNITARIO,KG_UNIDAD,0.1800,0.1950,<NA>,<NA>,<NA>,SI,PARAMETRIZADO
1,14003025,UNITARIO,KG_UNIDAD,0.1200,0.1300,<NA>,<NA>,<NA>,SI,PARAMETRIZADO
2,12002518,PRESENTACION,KG,<NA>,<NA>,2,<NA>,<NA>,SI,PARAMETRIZADO



tolerancias_consumo: 25 filas x 13 columnas
Columnas normalizadas: ['cod_producto', 'cod_componente', 'tipo_componente', 'metodo_validacion', 'tolerancia_inferior', 'tolerancia_superior', 'unidad_tolerancia', 'validar_individualmente', 'permite_merma', 'merma_maxima_porcentaje', 'genera_novedad', 'estado_parametrizacion', 'observacion']


,cod_producto,cod_componente,tipo_componente,metodo_validacion,tolerancia_inferior,tolerancia_superior,unidad_tolerancia,validar_individualmente,permite_merma,merma_maxima_porcentaje,genera_novedad,estado_parametrizacion,observacion
0,14002692,11001128,MP,PORCENTAJE,4,4,PORCENTAJE,SI,NO,<NA>,SI,PARAMETRIZADO,<NA>
1,14002692,12002544,MP,PORCENTAJE,3,3,PORCENTAJE,SI,NO,<NA>,SI,PARAMETRIZADO,<NA>
2,14002692,11001132,MP,PORCENTAJE,3,3,PORCENTAJE,SI,NO,<NA>,SI,PARAMETRIZADO,<NA>



reglas_especiales: 2 filas x 8 columnas
Columnas normalizadas: ['cod_producto', 'tipo_regla', 'valor_regla', 'unidad_regla', 'fecha_base', 'resultado_si_no_cumple', 'estado_parametrizacion', 'observacion']


,cod_producto,tipo_regla,valor_regla,unidad_regla,fecha_base,resultado_si_no_cumple,estado_parametrizacion,observacion
0,14002692,TEMPERATURA,2,DIAS_CALENDARIO,fecha_de_fin_real,Esperar control de temperatura,PARAMETRIZADO,<NA>
1,14003025,TEMPERATURA,2,DIAS_CALENDARIO,fecha_de_fin_real,Esperar control de temperatura,PARAMETRIZADO,<NA>


## 7. Validación previa a la consolidación

In [46]:
ESQUEMAS = {
    "cabeceras": ["orden", "material", "cantidad_entregada", "unidad_de_medida", "fecha_de_fin_real"],
    "consumos": ["orden", "material", "cantidad_movimiento", "clase_de_movimiento"],
    "productos": ["cod_producto", "unidad_sap", "unidades_por_presentacion", "peso_referencia_presentacion_kg"],
    "recetas": ["cod_producto", "cod_componente", "tipo_componente", "cantidad_receta", "unidad_cantidad_receta", "peso_estandar_kg"],
    "conversiones": ["cod_componente", "unidad_origen", "unidad_base_sap_real", "unidad_destino", "factor_a_unidad_destino", "factor_consumo_real_a_control", "tratamiento_consumo_real"],
    "tolerancias_peso": ["cod_producto", "nivel_control", "peso_minimo_unitario_kg", "peso_minimo_presentacion_kg"],
    "tolerancias_consumo": ["cod_producto", "cod_componente", "metodo_validacion"],
    "reglas_especiales": ["cod_producto", "tipo_regla", "valor_regla"],
}
LLAVES = {
    "productos": ["cod_producto"],
    "recetas": ["cod_producto", "cod_componente"],
    "conversiones": ["cod_componente", "unidad_origen", "unidad_destino"],
    "tolerancias_peso": ["cod_producto"],
    "tolerancias_consumo": ["cod_producto", "cod_componente"],
    "reglas_especiales": ["cod_producto", "tipo_regla"],
}

def validar_antes_de_consolidar(tablas: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    hallazgos = []
    for nombre, columnas in ESQUEMAS.items():
        for columna in sorted(set(columnas) - set(tablas[nombre].columns)):
            hallazgos.append({"nivel": "ERROR", "tabla": nombre, "regla": "COLUMNA_FALTANTE", "detalle": columna})
    for nombre, llave in LLAVES.items():
        cantidad = int(tablas[nombre].duplicated(llave, keep=False).sum())
        if cantidad:
            hallazgos.append({"nivel": "ERROR", "tabla": nombre, "regla": "LLAVE_DUPLICADA", "detalle": cantidad})

    return pd.DataFrame(hallazgos, columns=["nivel", "tabla", "regla", "detalle"])

HALLAZGOS = validar_antes_de_consolidar(TABLAS)

print("Resultado de la validación previa:")

if HALLAZGOS.empty:

    ESTADO_PARAMETRIZACION = "OK"

    display(pd.DataFrame([{
        "nivel": "OK",
        "tabla": "todas",
        "regla": "VALIDACION_COMPLETA",
        "detalle": "Sin errores críticos"
    }]))

else:

    ESTADO_PARAMETRIZACION = "ERROR"

    display(
        HALLAZGOS.sort_values(
            ["nivel", "tabla", "regla"]
        ).reset_index(drop=True)
    )

    print(
        "La consolidación no se ejecutará todavía. "
        "Revise los hallazgos mostrados."
    )
if HALLAZGOS.empty:
    display(pd.DataFrame([{"nivel": "OK", "tabla": "todas", "regla": "VALIDACION_COMPLETA", "detalle": "Sin errores críticos"}]))
else:
    display(HALLAZGOS)
    raise ValueError("Revise el diagnóstico mostrado antes de consolidar.")

Resultado de la validación previa:


,nivel,tabla,regla,detalle
0,OK,todas,VALIDACION_COMPLETA,Sin errores críticos


,nivel,tabla,regla,detalle
0,OK,todas,VALIDACION_COMPLETA,Sin errores críticos


## 8. Funciones de consolidación

In [47]:
def consolidar_movimientos(consumos: pd.DataFrame):
    """Consolida producción y consumo sin perder los signos SAP."""
    datos = consumos.copy()
    datos["clase_de_movimiento"] = datos["clase_de_movimiento"].astype("string").str.replace(r"\.0$", "", regex=True)
    es_produccion = datos["clase_de_movimiento"].isin(MOVIMIENTOS_PRODUCCION)
    es_consumo = datos["clase_de_movimiento"].isin(MOVIMIENTOS_CONSUMO)
    es_revision = datos["clase_de_movimiento"].isin(MOVIMIENTOS_REVISION)
    produccion = datos.loc[es_produccion].groupby("orden", as_index=False).agg(produccion_neta_sap=("cantidad_movimiento", "sum"))
    produccion["cantidad_producida_sap"] = -produccion["produccion_neta_sap"]
    consumo = (datos.loc[es_consumo].groupby(["orden", "material"], as_index=False)
               .agg(cantidad_real_sap=("cantidad_movimiento", "sum"), descripcion_material=("descripcion_material", "first"))
               .rename(columns={"material": "cod_componente"}))
    revision = datos.loc[es_revision].copy()
    return produccion, consumo, revision


def construir_ordenes(cabeceras, productos, produccion):
    codigos = set(productos["cod_producto"])
    ordenes = (cabeceras.loc[cabeceras["material"].isin(codigos)].copy()
               .rename(columns={"material": "cod_producto", "texto_breve_material": "descripcion_cabecera",
                                "unidad_de_medida": "unidad_sap_cabecera", "fecha_de_fin_real": "fecha_produccion"}))
    ordenes = (ordenes.merge(produccion, on="orden", validate="one_to_one")
               .merge(productos, on="cod_producto", validate="many_to_one", suffixes=("", "_param")))
    ordenes["presentaciones_producidas"] = ordenes["cantidad_producida_sap"]
    en_kg = (ordenes["unidad_sap_cabecera"].eq("KG") & ~ordenes["unidad_sap"].eq("KG") & ordenes["peso_referencia_presentacion_kg"].gt(0))
    ordenes.loc[en_kg, "presentaciones_producidas"] = ordenes.loc[en_kg, "cantidad_producida_sap"] / ordenes.loc[en_kg, "peso_referencia_presentacion_kg"]
    ordenes["unidades_producidas"] = ordenes["presentaciones_producidas"] * ordenes["unidades_por_presentacion"]
    return ordenes

## 9. Consolidación de órdenes y movimientos

In [48]:
PRODUCCION_NETA, CONSUMOS_NETOS, MOVIMIENTOS_REVISION = consolidar_movimientos(TABLAS["consumos"])
ORDENES = construir_ordenes(TABLAS["cabeceras"], TABLAS["productos"], PRODUCCION_NETA)
print("Órdenes parametrizadas:", len(ORDENES))
display(PRODUCCION_NETA.head())
display(CONSUMOS_NETOS.head())
display(ORDENES.head())

Órdenes parametrizadas: 33


,orden,produccion_neta_sap,cantidad_producida_sap
0,1082908,"-1,356.0000","1,356.0000"
1,1082915,"-1,360.0000","1,360.0000"
2,1082917,"-12,894.6000","12,894.6000"
3,1082918,"-2,280.0000","2,280.0000"
4,1082919,-375.0000,375.0000


,orden,cod_componente,cantidad_real_sap,descripcion_material
0,1082908,11001135,110.0000,MARINADO PICANTE CJX25UN
1,1082908,12002532,"1,844.1100",ALAS CORTADAS POLLO
2,1082908,18001585,100.0000,BOLSA FUNDA BTOX10BOLX100UN
3,1082908,18001586,"1,400.0000",BOLSA FUELLE BTOX50BOLX100UN
4,1082908,18001602,"1,400.0000",ETIQUETA TERMICA CJX4ROLX2605UN


,centro_planificacion,orden,almacen,cod_producto,lote,descripcion_cabecera,cantidad_orden,cantidad_entregada,fecha_produccion,unidad_sap_cabecera,estado_de_sistema,status_de_usuario,hora_creacion,fecha_inicio_extrema,fecha_fin_extrema,autor,planificador_nec,version_fabricacion,modificado_por,produccion_neta_sap,cantidad_producida_sap,descripcion_producto,unidad_sap,unidades_por_presentacion,peso_referencia_unitario_kg,peso_referencia_presentacion_kg,peso_aproximado,requiere_temperatura,dias_temperatura,estado_parametrizacion,presentaciones_producidas,unidades_producidas
0,B010,1082985,1008,14002692,2605190483,AVALANCHA OREO CJ(63UN),208.0000,208.0000,2026-05-22,CJ,CERR FMAT NOTP ENTR PREC EDET DDPN DMNV*,RECT,9:10:01 a. m.,2026-05-19,26/5/2026,USUARIO 1,P07,V001,CABAD,-208.0000,208.0000,AVALANCHA OREO CJ(63UN),CJ,63,0.1880,11.8440,SI,SI,2,PARAMETRIZADO,208.0000,"13,104.0000"
1,B010,1083164,1008,14002692,2605230408,AVALANCHA OREO CJ(63UN),80.0000,80.0000,2026-05-24,CJ,CERR NOTP ENTR PREC EDET DDPN DMNV LINA*,LIB RECT,8:57:41 a. m.,2026-05-23,25/5/2026,USUARIO 1,P07,V001,CABAD,-80.0000,80.0000,AVALANCHA OREO CJ(63UN),CJ,63,0.1880,11.8440,SI,SI,2,PARAMETRIZADO,80.0000,"5,040.0000"
2,B010,1083177,1008,14002692,2605230886,AVALANCHA OREO CJ(63UN),180.0000,180.0000,2026-05-25,CJ,CERR NOTP ENTR PREC EDET DDPN DMNV LINA*,LIB RECT,7:28:48 p. m.,2026-05-23,26/5/2026,USUARIO 1,P07,V001,CABAD,-180.0000,180.0000,AVALANCHA OREO CJ(63UN),CJ,63,0.1880,11.8440,SI,SI,2,PARAMETRIZADO,180.0000,"11,340.0000"
3,B010,1083022,1008,14003025,2605200468,SUNDAE DE CHOCOLATE CJ(105UN),120.0000,120.0000,2026-05-22,CJ,CERR NOTP ENTR PREC EDET DDPN DMNV LINA*,LIB RECT,9:14:19 a. m.,2026-05-20,23/5/2026,USUARIO 2,P07,V001,CABAD,-120.0000,120.0000,SUNDAE DE CHOCOLATE CJ(105UN),CJ,105,0.1250,13.1250,SI,SI,2,PARAMETRIZADO,120.0000,"12,600.0000"
4,B010,1082921,1001,14002427,2605180214,BIG CRUNCH MARINADO BOL(10UN),390.0000,390.0000,2026-05-18,BOL,CERR FMAT NOTP ENTR PREC EDET CONA DDPN*,LIB RECT,7:22:48 a. m.,2026-05-18,18/5/2026,USUARIO 6,P01,V001,CABAD,-390.0000,390.0000,BIG CRUNCH MARINADO BOL(10UN),BOL,10,0.0550,0.5500,SI,NO,0,PARAMETRIZADO,390.0000,"3,900.0000"


## 10. Función de detalle y tolerancias

In [49]:
def construir_detalle(ordenes, recetas, consumos, conversiones, tolerancias):
    detalle = (ordenes.merge(recetas, on="cod_producto", validate="many_to_many", suffixes=("", "_receta"))
               .merge(consumos, on=["orden", "cod_componente"], how="left", validate="one_to_one")
               .merge(conversiones, on="cod_componente", validate="many_to_one", suffixes=("", "_conversion")))
    es_mp = detalle["tipo_componente"].eq("MP")
    es_insumo = detalle["tipo_componente"].eq("IN")
    detalle["cantidad_esperada"] = np.nan
    detalle.loc[es_mp, "cantidad_esperada"] = detalle.loc[es_mp, "peso_estandar_kg"] * detalle.loc[es_mp, "presentaciones_producidas"]
    detalle.loc[es_insumo, "cantidad_esperada"] = detalle.loc[es_insumo, "cantidad_receta"] * detalle.loc[es_insumo, "presentaciones_producidas"]
    detalle["cantidad_real"] = detalle["cantidad_real_sap"]
    convertir = detalle["tratamiento_consumo_real"].eq("CONVERTIR_A_UNIDAD_CONTROL")
    detalle.loc[convertir, "cantidad_real"] = detalle.loc[convertir, "cantidad_real_sap"] * detalle.loc[convertir, "factor_consumo_real_a_control"]

    detalle = detalle.merge(tolerancias.rename(columns={"estado_parametrizacion": "estado_tolerancia"}),
                            on=["cod_producto", "cod_componente"], validate="many_to_one", suffixes=("", "_tol"))
    detalle["limite_inferior"] = np.nan
    detalle["limite_superior"] = np.nan
    metodo = detalle["metodo_validacion"].str.upper()
    porcentaje = metodo.eq("PORCENTAJE")
    cantidad = metodo.eq("CANTIDAD")
    presencia = metodo.eq("PRESENCIA")
    merma = detalle["permite_merma"].str.upper().eq("SI")
    detalle.loc[porcentaje, "limite_inferior"] = detalle.loc[porcentaje, "cantidad_esperada"] * (1 - detalle.loc[porcentaje, "tolerancia_inferior"].fillna(0) / 100)
    detalle.loc[porcentaje & ~merma, "limite_superior"] = detalle.loc[porcentaje & ~merma, "cantidad_esperada"] * (1 + detalle.loc[porcentaje & ~merma, "tolerancia_superior"].fillna(0) / 100)
    detalle.loc[porcentaje & merma, "limite_superior"] = detalle.loc[porcentaje & merma, "cantidad_esperada"] * (1 + detalle.loc[porcentaje & merma, "merma_maxima_porcentaje"] / 100)
    detalle.loc[cantidad, "limite_inferior"] = detalle.loc[cantidad, "cantidad_esperada"] - detalle.loc[cantidad, "tolerancia_inferior"].fillna(0)
    detalle.loc[cantidad, "limite_superior"] = detalle.loc[cantidad, "cantidad_esperada"] + detalle.loc[cantidad, "tolerancia_superior"].fillna(0)
    detalle.loc[presencia, "limite_inferior"] = 1e-12
    detalle.loc[presencia, "limite_superior"] = np.inf

    ruta = detalle["unidad_cantidad_receta"].eq(detalle["unidad_origen"]) | detalle["unidad_cantidad_receta"].eq(detalle["unidad_destino"])
    pendiente = detalle["metodo_validacion"].isna() | ~ruta | (es_mp & detalle["peso_estandar_kg"].isna()) | (porcentaje & merma & detalle["merma_maxima_porcentaje"].isna())
    detalle["tipo_novedad"] = "Sin novedad"
    detalle.loc[pendiente, "tipo_novedad"] = "Revisión manual por parametrización pendiente"
    valida = ~pendiente
    faltante = valida & es_insumo & detalle["componente_obligatorio"].str.upper().eq("SI") & detalle["cantidad_real"].fillna(0).le(0)
    detalle.loc[faltante, "tipo_novedad"] = "Reportar por insumo faltante"
    detalle.loc[valida & ~faltante & detalle["cantidad_real"].fillna(0).lt(detalle["limite_inferior"]), "tipo_novedad"] = "Reportar por consumo inferior"
    detalle.loc[valida & ~merma & detalle["cantidad_real"].gt(detalle["limite_superior"]), "tipo_novedad"] = "Reportar por consumo superior"
    detalle.loc[valida & merma & detalle["cantidad_real"].gt(detalle["limite_superior"]), "tipo_novedad"] = "Reportar por merma superior"
    detalle["diferencia"] = detalle["cantidad_real"].fillna(0) - detalle["cantidad_esperada"]
    detalle["porcentaje_desviacion"] = np.where(detalle["cantidad_esperada"].ne(0), detalle["diferencia"] / detalle["cantidad_esperada"] * 100, np.nan)
    return detalle, ruta, pendiente

## 11. Detalle orden-receta

In [50]:
DETALLE, RUTAS_VALIDAS, PENDIENTES_RECETA = construir_detalle(
    ORDENES, TABLAS["recetas"], CONSUMOS_NETOS, TABLAS["conversiones"], TABLAS["tolerancias_consumo"]
)
print("Filas orden-receta:", len(DETALLE))
display(DETALLE[["orden", "fecha_produccion", "cod_producto", "cod_componente", "cantidad_esperada", "cantidad_real", "diferencia", "tipo_novedad"]].head(20))

Filas orden-receta: 97


,orden,fecha_produccion,cod_producto,cod_componente,cantidad_esperada,cantidad_real,diferencia,tipo_novedad
0,1082985,2026-05-22,14002692,11001128,"1,976.8320","1,975.0000",-1.8320,Sin novedad
1,1082985,2026-05-22,14002692,12002544,230.4016,238.0200,7.6184,Reportar por consumo superior
2,1082985,2026-05-22,14002692,11001132,262.0800,210.4800,-51.6000,Reportar por consumo inferior
3,1082985,2026-05-22,14002692,18001665,832.0000,624.0000,-208.0000,Reportar por consumo inferior
4,1082985,2026-05-22,14002692,18001501,"13,104.0000","13,155.0000",51.0000,Reportar por consumo superior
5,1082985,2026-05-22,14002692,15005756,"13,104.0000","9,200.0000","-3,904.0000",Reportar por consumo inferior
6,1082985,2026-05-22,14002692,18001664,208.0000,208.0000,0.0000,Sin novedad
7,1083164,2026-05-24,14002692,11001128,760.3200,756.0000,-4.3200,Sin novedad
8,1083164,2026-05-24,14002692,12002544,88.6160,91.6800,3.0640,Reportar por consumo superior
9,1083164,2026-05-24,14002692,11001132,100.8000,85.5800,-15.2200,Reportar por consumo inferior


## 12. Peso y componentes no parametrizados

In [51]:
def calcular_peso(detalle, ordenes, tolerancias_peso):
    es_mp = detalle["tipo_componente"].eq("MP")
    detalle = detalle.copy()
    detalle["peso_real_componente_kg"] = pd.Series(np.nan, index=detalle.index, dtype="Float64")
    detalle.loc[es_mp, "peso_real_componente_kg"] = detalle.loc[es_mp, "cantidad_real_sap"]
    peso = detalle.loc[es_mp].groupby(["orden", "cod_producto"], as_index=False).agg(
        peso_total_real_kg=("peso_real_componente_kg", lambda s: s.sum(min_count=1)))
    peso = (peso.merge(ordenes[["orden", "cod_producto", "fecha_produccion", "descripcion_producto", "presentaciones_producidas", "unidades_producidas"]],
                       on=["orden", "cod_producto"], validate="one_to_one")
            .merge(tolerancias_peso, on="cod_producto", validate="many_to_one"))
    peso["peso_unitario_real_kg"] = peso["peso_total_real_kg"] / peso["unidades_producidas"]
    peso["peso_presentacion_real_kg"] = peso["peso_total_real_kg"] / peso["presentaciones_producidas"]
    unitario = peso["nivel_control"].eq("UNITARIO")
    peso["valor_control"] = np.where(unitario, peso["peso_unitario_real_kg"], peso["peso_presentacion_real_kg"])
    peso["limite_inferior"] = np.where(unitario, peso["peso_minimo_unitario_kg"], peso["peso_minimo_presentacion_kg"])
    peso["limite_superior"] = np.where(unitario, peso["peso_maximo_unitario_kg"], peso["peso_maximo_presentacion_kg"])
    peso["resultado_peso"] = "Peso dentro del rango"
    peso.loc[peso["valor_control"].lt(peso["limite_inferior"]), "resultado_peso"] = "Peso inferior"
    peso.loc[peso["limite_superior"].notna() & peso["valor_control"].gt(peso["limite_superior"]), "resultado_peso"] = "Peso superior"
    return peso

PESO = calcular_peso(DETALLE, ORDENES, TABLAS["tolerancias_peso"])

LLAVES_RECETA = set(zip(TABLAS["recetas"]["cod_producto"], TABLAS["recetas"]["cod_componente"]))
REALES = CONSUMOS_NETOS.merge(ORDENES[["orden", "cod_producto", "fecha_produccion"]].drop_duplicates("orden"), on="orden")
REALES["parametrizado"] = [(p, c) in LLAVES_RECETA for p, c in zip(REALES["cod_producto"], REALES["cod_componente"])]
NO_PARAMETRIZADOS = REALES.loc[~REALES["parametrizado"]].copy()
NO_PARAMETRIZADOS["estado"] = "Advertencia de parametrización"
NO_PARAMETRIZADOS["motivo"] = "Componente consumido sin estándar de receta"

display(PESO.head(15))
display(NO_PARAMETRIZADOS.head(15))

,orden,cod_producto,peso_total_real_kg,fecha_produccion,descripcion_producto,presentaciones_producidas,unidades_producidas,nivel_control,unidad_control,peso_minimo_unitario_kg,peso_maximo_unitario_kg,peso_minimo_presentacion_kg,peso_maximo_presentacion_kg,tolerancia_porcentaje,genera_novedad,estado_parametrizacion,peso_unitario_real_kg,peso_presentacion_real_kg,valor_control,limite_inferior,limite_superior,resultado_peso
0,1082919,14002441,367.5000,2026-05-18,STRIPS MARINADO BOL(20UN),375.0000,"7,500.0000",UNITARIO,KG_UNIDAD,0.0450,0.0550,<NA>,<NA>,<NA>,SI,PARAMETRIZADO,0.0490,0.9800,0.0490,0.0450,0.0550,Peso dentro del rango
1,1082921,14002427,210.6000,2026-05-18,BIG CRUNCH MARINADO BOL(10UN),390.0000,"3,900.0000",UNITARIO,KG_UNIDAD,0.0550,0.0600,<NA>,<NA>,<NA>,SI,PARAMETRIZADO,0.0540,0.5400,0.0540,0.0550,0.0600,Peso inferior
2,1082931,14002441,204.5000,2026-05-18,STRIPS MARINADO BOL(20UN),247.0000,"4,940.0000",UNITARIO,KG_UNIDAD,0.0450,0.0550,<NA>,<NA>,<NA>,SI,PARAMETRIZADO,0.0414,0.8279,0.0414,0.0450,0.0550,Peso inferior
3,1082933,14002427,740.1200,2026-05-18,BIG CRUNCH MARINADO BOL(10UN),"1,441.0000","14,410.0000",UNITARIO,KG_UNIDAD,0.0550,0.0600,<NA>,<NA>,<NA>,SI,PARAMETRIZADO,0.0514,0.5136,0.0514,0.0550,0.0600,Peso inferior
4,1082941,14002428,689.1000,2026-05-19,LECHUGA BATAVIA PROCESADA BOL(1KG),337.0000,337.0000,PRESENTACION,KG,<NA>,<NA>,1,<NA>,<NA>,SI,PARAMETRIZADO,2.0448,2.0448,2.0448,1.0000,NaN,Peso dentro del rango
5,1082952,12002518,"1,115.1880",2026-05-19,MIX VERDURAS ENSALADA KFC,482.1000,482.1000,PRESENTACION,KG,<NA>,<NA>,2,<NA>,<NA>,SI,PARAMETRIZADO,2.3132,2.3132,2.3132,2.0000,NaN,Peso dentro del rango
6,1082963,14002427,318.1000,2026-05-19,BIG CRUNCH MARINADO BOL(10UN),548.0000,"5,480.0000",UNITARIO,KG_UNIDAD,0.0550,0.0600,<NA>,<NA>,<NA>,SI,PARAMETRIZADO,0.0580,0.5805,0.0580,0.0550,0.0600,Peso dentro del rango
7,1082965,14002441,133.0000,2026-05-19,STRIPS MARINADO BOL(20UN),405.0000,"8,100.0000",UNITARIO,KG_UNIDAD,0.0450,0.0550,<NA>,<NA>,<NA>,SI,PARAMETRIZADO,0.0164,0.3284,0.0164,0.0450,0.0550,Peso inferior
8,1082985,14002692,"2,423.5000",2026-05-22,AVALANCHA OREO CJ(63UN),208.0000,"13,104.0000",UNITARIO,KG_UNIDAD,0.1800,0.1950,<NA>,<NA>,<NA>,SI,PARAMETRIZADO,0.1849,11.6514,0.1849,0.1800,0.1950,Peso dentro del rango
9,1082986,14002441,420.5000,2026-05-19,STRIPS MARINADO BOL(20UN),100.0000,"2,000.0000",UNITARIO,KG_UNIDAD,0.0450,0.0550,<NA>,<NA>,<NA>,SI,PARAMETRIZADO,0.2102,4.2050,0.2102,0.0450,0.0550,Peso superior


,orden,cod_componente,cantidad_real_sap,descripcion_material,cod_producto,fecha_produccion,parametrizado,estado,motivo
0,1082919,11000092,0.1720,CARRAGENINA BTOX25KG,14002441,2026-05-18,False,Advertencia de parametrización,Componente consumido sin estándar de receta
1,1082919,11000341,5.0000,MARINADOR CRISPY 15.66KG CJX60UN,14002441,2026-05-18,False,Advertencia de parametrización,Componente consumido sin estándar de receta
2,1082919,11001130,3.1580,SAL YODADA BTOX25UNX0.5KG,14002441,2026-05-18,False,Advertencia de parametrización,Componente consumido sin estándar de receta
4,1082919,18001585,15.0000,BOLSA FUNDA BTOX10BOLX100UN,14002441,2026-05-18,False,Advertencia de parametrización,Componente consumido sin estándar de receta
6,1082919,18001602,400.0000,ETIQUETA TERMICA CJX4ROLX2605UN,14002441,2026-05-18,False,Advertencia de parametrización,Componente consumido sin estándar de receta
7,1082919,18001612,"1,245.8000",CINTA DE RESINA 110*450M ROLX45000CM,14002441,2026-05-18,False,Advertencia de parametrización,Componente consumido sin estándar de receta
8,1082921,11000341,5.0000,MARINADOR CRISPY 15.66KG CJX60UN,14002427,2026-05-18,False,Advertencia de parametrización,Componente consumido sin estándar de receta
10,1082921,18001585,20.0000,BOLSA FUNDA BTOX10BOLX100UN,14002427,2026-05-18,False,Advertencia de parametrización,Componente consumido sin estándar de receta
11,1082921,18001586,400.0000,BOLSA FUELLE BTOX50BOLX100UN,14002427,2026-05-18,False,Advertencia de parametrización,Componente consumido sin estándar de receta
12,1082921,18001602,500.0000,ETIQUETA TERMICA CJX4ROLX2605UN,14002427,2026-05-18,False,Advertencia de parametrización,Componente consumido sin estándar de receta


## 13. Pruebas internas

In [52]:
pruebas = []
def probar(nombre, condicion, valor):
    pruebas.append({"prueba": nombre, "resultado": "OK" if bool(condicion) else "ERROR", "valor": valor})

ordenes_esperadas = TABLAS["cabeceras"].loc[TABLAS["cabeceras"]["material"].isin(set(TABLAS["productos"]["cod_producto"])), "orden"].nunique()
detalle_esperado = sum(TABLAS["cabeceras"].loc[TABLAS["cabeceras"]["material"].isin(set(TABLAS["productos"]["cod_producto"])), "material"].value_counts().get(cod, 0) * n
                       for cod, n in TABLAS["recetas"].groupby("cod_producto").size().items())
probar("Órdenes dinámicas", len(ORDENES) == ordenes_esperadas, f"{len(ORDENES)}/{ordenes_esperadas}")
probar("Detalle dinámico", len(DETALLE) == detalle_esperado, f"{len(DETALLE)}/{detalle_esperado}")
probar("Órdenes sin duplicados", not ORDENES["orden"].duplicated().any(), int(ORDENES["orden"].duplicated().sum()))
probar("Conversiones sin duplicados", not TABLAS["conversiones"].duplicated(["cod_componente", "unidad_origen", "unidad_destino"]).any(), 0)
probar("Rutas válidas", RUTAS_VALIDAS.all(), int((~RUTAS_VALIDAS).sum()))
probar("Sin pendientes de receta", not PENDIENTES_RECETA.any(), int(PENDIENTES_RECETA.sum()))
MOVIMIENTOS_REVISION_MUESTRA = MOVIMIENTOS_REVISION.loc[MOVIMIENTOS_REVISION["orden"].isin(ORDENES["orden"])].copy()
probar("Sin movimientos 531/532 en la muestra",MOVIMIENTOS_REVISION_MUESTRA.empty,len(MOVIMIENTOS_REVISION_MUESTRA))
tapa = DETALLE.loc[DETALLE["cod_componente"].eq("15005756")]
probar("Tapa BTO a UN por 50", (tapa["cantidad_real"] == tapa["cantidad_real_sap"] * 50).all(), len(tapa))
probar("Pesos no negativos", PESO["peso_total_real_kg"].ge(0).all(), PESO["peso_total_real_kg"].min())
PRUEBAS_INTERNAS = pd.DataFrame(pruebas)
display(PRUEBAS_INTERNAS)
PRUEBAS_CON_ERROR = PRUEBAS_INTERNAS.loc[
    PRUEBAS_INTERNAS["resultado"].eq("ERROR")
].copy()

if PRUEBAS_CON_ERROR.empty:
    print("Todas las pruebas internas finalizaron correctamente.")
else:
    print("Se encontraron pruebas con error:")
    display(PRUEBAS_CON_ERROR)

,prueba,resultado,valor
0,Órdenes dinámicas,OK,33/33
1,Detalle dinámico,OK,97/97
2,Órdenes sin duplicados,OK,0
3,Conversiones sin duplicados,OK,0
4,Rutas válidas,OK,0
5,Sin pendientes de receta,OK,0
6,Sin movimientos 531/532 en la muestra,OK,0
7,Tapa BTO a UN por 50,OK,3
8,Pesos no negativos,OK,133.0000


Todas las pruebas internas finalizaron correctamente.


## 14. Resumen e indicadores

In [53]:
def unir_novedades(serie):
    valores = sorted(set(serie.dropna()) - {"Sin novedad"})
    return " | ".join(valores) if valores else "Sin novedad"

MP_RES = DETALLE.loc[DETALLE["tipo_componente"].eq("MP")].groupby("orden")["tipo_novedad"].apply(unir_novedades).rename("resultado_materias_primas")
IN_RES = DETALLE.loc[DETALLE["tipo_componente"].eq("IN")].groupby("orden")["tipo_novedad"].apply(unir_novedades).rename("resultado_insumos")
ADV = NO_PARAMETRIZADOS.groupby("orden").agg(cantidad_componentes_no_parametrizados=("cod_componente", "nunique"), advertencias=("descripcion_material", lambda s: " | ".join(sorted(set(s)))))
RESUMEN = ORDENES[["orden", "fecha_produccion", "cod_producto", "descripcion_producto", "cantidad_producida_sap"]].drop_duplicates("orden")
RESUMEN = RESUMEN.merge(PESO[["orden", "resultado_peso"]], on="orden").merge(MP_RES, on="orden").merge(IN_RES, on="orden").merge(ADV, on="orden", how="left")
RESUMEN["cantidad_componentes_no_parametrizados"] = RESUMEN["cantidad_componentes_no_parametrizados"].fillna(0).astype(int)
RESUMEN["tiene_advertencia_parametrizacion"] = np.where(RESUMEN["cantidad_componentes_no_parametrizados"].gt(0), "SI", "NO")
RESUMEN["advertencias"] = RESUMEN["advertencias"].fillna("")

def motivos(fila):
    salida = []
    if fila["resultado_peso"] == "Peso inferior": salida.append("Reportar por peso inferior")
    if fila["resultado_peso"] == "Peso superior": salida.append("Reportar por peso superior")
    for c in ["resultado_materias_primas", "resultado_insumos"]:
        if fila[c] != "Sin novedad": salida += fila[c].split(" | ")
    return sorted(set(salida))
RESUMEN["lista_motivos"] = RESUMEN.apply(motivos, axis=1)
RESUMEN["motivos"] = RESUMEN["lista_motivos"].str.join(" | ")
RESUMEN["estado_general"] = RESUMEN["lista_motivos"].apply(lambda x: " | ".join(x) if x else "Validada")
RESUMEN["recomendacion"] = np.where(RESUMEN["estado_general"].eq("Validada"), "Continuar cierre y liberación", "Reportar a Producción y Costos")
INDICADORES = pd.DataFrame({"indicador":["Total de órdenes","Órdenes validadas","Órdenes para reportar","Órdenes con advertencia"],"valor":[RESUMEN.orden.nunique(),RESUMEN.loc[RESUMEN.estado_general.eq("Validada"),"orden"].nunique(),RESUMEN.loc[RESUMEN.estado_general.str.contains("Reportar"),"orden"].nunique(),RESUMEN.loc[RESUMEN.tiene_advertencia_parametrizacion.eq("SI"),"orden"].nunique()]})
display(INDICADORES)
display(RESUMEN.head(20))

,indicador,valor
0,Total de órdenes,24
1,Órdenes validadas,0
2,Órdenes para reportar,24
3,Órdenes con advertencia,24


,orden,fecha_produccion,cod_producto,descripcion_producto,cantidad_producida_sap,resultado_peso,resultado_materias_primas,resultado_insumos,cantidad_componentes_no_parametrizados,advertencias,tiene_advertencia_parametrizacion,lista_motivos,motivos,estado_general,recomendacion
0,1082985,2026-05-22,14002692,AVALANCHA OREO CJ(63UN),208.0000,Peso dentro del rango,Reportar por consumo inferior | Reportar por c...,Reportar por consumo inferior | Reportar por c...,4,CINTA DE RESINA 110*450M ROLX45000CM | ETIQUET...,SI,"[Reportar por consumo inferior, Reportar por c...",Reportar por consumo inferior | Reportar por c...,Reportar por consumo inferior | Reportar por c...,Reportar a Producción y Costos
1,1083164,2026-05-24,14002692,AVALANCHA OREO CJ(63UN),80.0000,Peso dentro del rango,Reportar por consumo inferior | Reportar por c...,Reportar por consumo inferior,3,CINTA DE RESINA 110*450M ROLX45000CM | ETIQUET...,SI,"[Reportar por consumo inferior, Reportar por c...",Reportar por consumo inferior | Reportar por c...,Reportar por consumo inferior | Reportar por c...,Reportar a Producción y Costos
2,1083177,2026-05-25,14002692,AVALANCHA OREO CJ(63UN),180.0000,Peso dentro del rango,Reportar por consumo inferior,Reportar por consumo inferior | Reportar por c...,2,ETIQUETA POSTRES CJX6ROLX10786UN | ETIQUETA TE...,SI,"[Reportar por consumo inferior, Reportar por c...",Reportar por consumo inferior | Reportar por c...,Reportar por consumo inferior | Reportar por c...,Reportar a Producción y Costos
3,1083022,2026-05-22,14003025,SUNDAE DE CHOCOLATE CJ(105UN),120.0000,Peso dentro del rango,Reportar por consumo inferior,Reportar por consumo inferior | Reportar por c...,3,CINTA DE RESINA 110*450M ROLX45000CM | ETIQUET...,SI,"[Reportar por consumo inferior, Reportar por c...",Reportar por consumo inferior | Reportar por c...,Reportar por consumo inferior | Reportar por c...,Reportar a Producción y Costos
4,1082919,2026-05-18,14002441,STRIPS MARINADO BOL(20UN),375.0000,Peso dentro del rango,Reportar por consumo superior,Reportar por consumo superior,6,BOLSA FUNDA BTOX10BOLX100UN | CARRAGENINA BTOX...,SI,[Reportar por consumo superior],Reportar por consumo superior,Reportar por consumo superior,Reportar a Producción y Costos
5,1082931,2026-05-18,14002441,STRIPS MARINADO BOL(20UN),247.0000,Peso inferior,Reportar por consumo superior,Reportar por consumo superior,6,BOLSA FUNDA BTOX10BOLX100UN | CARRAGENINA BTOX...,SI,"[Reportar por consumo superior, Reportar por p...",Reportar por consumo superior | Reportar por p...,Reportar por consumo superior | Reportar por p...,Reportar a Producción y Costos
6,1082965,2026-05-19,14002441,STRIPS MARINADO BOL(20UN),405.0000,Peso inferior,Reportar por consumo superior,Reportar por consumo superior,6,BOLSA FUNDA BTOX10BOLX100UN | CARRAGENINA BTOX...,SI,"[Reportar por consumo superior, Reportar por p...",Reportar por consumo superior | Reportar por p...,Reportar por consumo superior | Reportar por p...,Reportar a Producción y Costos
7,1082986,2026-05-19,14002441,STRIPS MARINADO BOL(20UN),100.0000,Peso superior,Reportar por consumo superior,Reportar por consumo superior,6,BOLSA FUNDA BTOX10BOLX100UN | CARRAGENINA BTOX...,SI,"[Reportar por consumo superior, Reportar por p...",Reportar por consumo superior | Reportar por p...,Reportar por consumo superior | Reportar por p...,Reportar a Producción y Costos
8,1083018,2026-05-20,14002441,STRIPS MARINADO BOL(20UN),623.0000,Peso dentro del rango,Reportar por consumo superior,Reportar por consumo superior,7,BOLSA FUNDA PERFORADA BTOX10BOLX100UN | CARRAG...,SI,[Reportar por consumo superior],Reportar por consumo superior,Reportar por consumo superior,Reportar a Producción y Costos
9,1083020,2026-05-20,14002441,STRIPS MARINADO BOL(20UN),150.0000,Peso dentro del rango,Reportar por consumo superior,Reportar por consumo superior,6,BOLSA FUNDA BTOX10BOLX100UN | CARRAGENINA BTOX...,SI,[Reportar por consumo superior],Reportar por consumo superior,Reportar por consumo superior,Reportar a 

## 15. Gráficos interactivos por fecha

In [56]:

  # ============================================================
# 15. GRÁFICOS INTERACTIVOS DE CONTROL
# ============================================================

RUTA_SALIDA = Path("resultados_actividad_5")

# Crea la carpeta si no existe.
RUTA_SALIDA.mkdir(parents=True,exist_ok=True)

print("Directorio actual:", Path.cwd())
print("Carpeta de salida:", RUTA_SALIDA.resolve())
print("¿Existe la carpeta?:", RUTA_SALIDA.exists())

GRUPOS_PRODUCTOS = {
    "HELADOS": ["14002692", "14003025"],
    "CARNES": ["14002427", "14002441"],
    "VERDURAS": ["12002518", "14002428"]
}

RUTAS_GRAFICOS = {}


# ============================================================
# FUNCIÓN 1. CONSOLIDAR PESO POR PRODUCTO Y FECHA
# ============================================================

def consolidar_peso_diario(datos, tipo_producto):
    """
    Consolida todas las órdenes del mismo producto y fecha.

    Helados y carnes:
    peso diario = peso total real / unidades producidas.

    Verduras:
    peso diario = peso total real / presentaciones producidas.
    """

    datos = datos.copy()

    columnas_requeridas = [
        "cod_producto",
        "descripcion_producto",
        "fecha_produccion",
        "orden",
        "peso_total_real_kg",
        "unidades_producidas",
        "presentaciones_producidas",
        "limite_inferior",
        "limite_superior"
    ]

    columnas_faltantes = [columna
        for columna in columnas_requeridas
        if columna not in datos.columns
    ]

    if columnas_faltantes:

        raise ValueError("Faltan columnas en PESO: "+ ", ".join(columnas_faltantes))

    diario = (
        datos.groupby(
            [
                "cod_producto",
                "descripcion_producto",
                "fecha_produccion"
            ],
            as_index=False,
            dropna=False
        )
        .agg(
            peso_total_real_kg=(
                "peso_total_real_kg",
                "sum"
            ),
            unidades_producidas=(
                "unidades_producidas",
                "sum"
            ),
            presentaciones_producidas=(
                "presentaciones_producidas",
                "sum"
            ),
            limite_inferior=(
                "limite_inferior",
                "first"
            ),
            limite_superior=(
                "limite_superior",
                "first"
            ),
            ordenes=(
                "orden",
                lambda serie: " | ".join(
                    sorted(
                        serie.dropna()
                        .astype(str)
                        .unique()
                    )
                )
            ),
            cantidad_ordenes=(
                "orden",
                "nunique"
            )
        )
    )

    if tipo_producto in ["HELADOS", "CARNES"]:

        diario["valor_control"] = np.where(
            diario["unidades_producidas"].gt(0),
            diario["peso_total_real_kg"]
            / diario["unidades_producidas"],
            np.nan
        )

    elif tipo_producto == "VERDURAS":

        diario["valor_control"] = np.where(
            diario["presentaciones_producidas"].gt(0),
            diario["peso_total_real_kg"]
            / diario["presentaciones_producidas"],
            np.nan
        )

    else:

        raise ValueError(f"Tipo de producto no reconocido: "f"{tipo_producto}")

    diario["resultado_peso"] = ("Peso dentro del rango")

    mascara_inferior = (diario["valor_control"]< diario["limite_inferior"])

    diario.loc[mascara_inferior,"resultado_peso"] = "Peso inferior"

    mascara_superior = (diario["limite_superior"].notna() & diario["valor_control"].gt(diario["limite_superior"])    )

    diario.loc[mascara_superior,"resultado_peso"] = "Peso superior"

    return (diario.sort_values(["cod_producto","fecha_produccion"]).reset_index(drop=True))

# ============================================================
# FUNCIÓN 2. CREAR GRÁFICO DE CONTROL
# ============================================================

def crear_grafico_control(
    datos,
    titulo,
    unidad_medida,
    multiplicador=1
):
    """
    Genera un gráfico de control con un solo punto
    por producto y fecha.
    """

    grafico = datos.copy()

    columnas_requeridas = [
        "fecha_produccion",
        "valor_control",
        "limite_inferior",
        "limite_superior",
        "resultado_peso"
    ]

    columnas_faltantes = [
        columna
        for columna in columnas_requeridas
        if columna not in grafico.columns
    ]

    if columnas_faltantes:

        raise ValueError(
            "Faltan columnas para crear el gráfico: "
            + ", ".join(columnas_faltantes)
        )

    # Preparamos las órdenes que aparecerán en el cursor.
    if "ordenes" in grafico.columns:

        grafico["ordenes_mostradas"] = (
            grafico["ordenes"]
            .fillna("Sin orden")
            .astype(str)
        )

    elif "orden" in grafico.columns:

        grafico["ordenes_mostradas"] = (
            grafico["orden"]
            .fillna("Sin orden")
            .astype(str)
        )

    else:

        grafico["ordenes_mostradas"] = (
            "Sin orden"
        )

    if "cantidad_ordenes" not in grafico.columns:

        grafico["cantidad_ordenes"] = 1

    grafico["valor_grafico"] = (
        grafico["valor_control"]
        * multiplicador
    )

    grafico["limite_inferior_grafico"] = (
        grafico["limite_inferior"]
        * multiplicador
    )

    grafico["limite_superior_grafico"] = (
        grafico["limite_superior"]
        * multiplicador
    )

    colores = {
        "Peso dentro del rango": "#2E8B57",
        "Peso inferior": "#F39C12",
        "Peso superior": "#D62728"
    }

    grafico["color_resultado"] = (
        grafico["resultado_peso"]
        .map(colores)
        .fillna("#808080")
    )

    customdata = np.stack(
        [
            grafico["ordenes_mostradas"],
            grafico["cantidad_ordenes"].astype(str),
            grafico["resultado_peso"].astype(str)
        ],
        axis=-1
    )

    figura = go.Figure()

    figura.add_trace(
        go.Scatter(
            x=grafico["fecha_produccion"],
            y=grafico["valor_grafico"],
            mode="lines+markers",
            name="Peso real",
            line=dict(
                color="#3366CC",
                width=2
            ),
            marker=dict(
                size=11,
                color=grafico["color_resultado"],
                line=dict(
                    color="white",
                    width=1
                )
            ),
            customdata=customdata,
            hovertemplate=(
                "Fecha: %{x|%d/%m/%Y}<br>"
                f"Peso: %{{y:.2f}} "
                f"{unidad_medida}<br>"
                "Órdenes: %{customdata[0]}<br>"
                "Cantidad de órdenes: "
                "%{customdata[1]}<br>"
                "Resultado: %{customdata[2]}"
                "<extra></extra>"
            )
        )
    )

    figura.add_trace(
        go.Scatter(
            x=grafico["fecha_produccion"],
            y=grafico[
                "limite_inferior_grafico"
            ],
            mode="lines",
            name="Límite inferior",
            line=dict(
                color="#F39C12",
                width=2,
                dash="dash"
            ),
            hovertemplate=(
                "Fecha: %{x|%d/%m/%Y}<br>"
                f"Límite inferior: %{{y:.2f}} "
                f"{unidad_medida}"
                "<extra></extra>"
            )
        )
    )

    if grafico[
        "limite_superior_grafico"
    ].notna().any():

        figura.add_trace(
            go.Scatter(
                x=grafico["fecha_produccion"],
                y=grafico[
                    "limite_superior_grafico"
                ],
                mode="lines",
                name="Límite superior",
                line=dict(
                    color="#D62728",
                    width=2,
                    dash="dash"
                ),
                hovertemplate=(
                    "Fecha: %{x|%d/%m/%Y}<br>"
                    f"Límite superior: %{{y:.2f}} "
                    f"{unidad_medida}"
                    "<extra></extra>"
                )
            )
        )

    figura.update_layout(
        title=titulo,
        xaxis_title="Fecha de producción",
        yaxis_title=unidad_medida,
        hovermode="x unified",
        template="plotly_white",
        legend_title="Control de peso",
        height=500
    )

    figura.update_xaxes(
        tickformat="%d/%m/%Y",
        type="date"
    )

    return figura


# ============================================================
# FUNCIÓN 3. GENERAR GRÁFICOS POR GRUPO
# ============================================================

def generar_graficos_grupo(
    grupo,
    codigos,
    unidad_medida,
    multiplicador
):
    """
    Genera los gráficos de todos los productos
    incluidos en un grupo.
    """

    print(
        f"GRÁFICOS DE CONTROL: {grupo}"
    )

    resultados_diarios = []

    for codigo in codigos:

        datos_producto = PESO.loc[
            PESO["cod_producto"].eq(codigo)
        ].copy()

        if datos_producto.empty:

            print(
                f"Sin información para el producto "
                f"{codigo}"
            )

            continue

        datos_diarios = (
            consolidar_peso_diario(
                datos=datos_producto,
                tipo_producto=grupo
            )
        )

        descripcion = datos_diarios[
            "descripcion_producto"
        ].iloc[0]

        if grupo == "HELADOS":

            subtitulo = (
                "Peso promedio diario por helado"
            )

            prefijo = "helado"

        elif grupo == "CARNES":

            subtitulo = (
                "Peso promedio diario por pieza"
            )

            prefijo = "carne"

        else:

            subtitulo = (
                "Peso promedio diario "
                "por presentación"
            )

            prefijo = "verdura"

        figura = crear_grafico_control(
            datos=datos_diarios,
            titulo=(
                f"{grupo.title()}: "
                f"{descripcion}<br>"
                f"{subtitulo}"
            ),
            unidad_medida=unidad_medida,
            multiplicador=multiplicador
        )

        figura.show()

        nombre_archivo = (
            f"grafico_{prefijo}_"
            f"{codigo}.html"
        )

        figura.write_html(
            RUTA_SALIDA / nombre_archivo,
            include_plotlyjs="cdn"
        )

        RUTAS_GRAFICOS[
            f"{prefijo}_{codigo}"
        ] = nombre_archivo

        datos_diarios[
            "grupo_producto"
        ] = grupo

        resultados_diarios.append(
            datos_diarios
        )

    if resultados_diarios:

        return pd.concat(
            resultados_diarios,
            ignore_index=True
        )

    return pd.DataFrame()


# ============================================================
# 15.1 HELADOS: GRAMOS POR HELADO
# ============================================================

PESO_DIARIO_HELADOS = (
    generar_graficos_grupo(
        grupo="HELADOS",
        codigos=GRUPOS_PRODUCTOS[
            "HELADOS"
        ],
        unidad_medida="gramos por helado",
        multiplicador=1000
    )
)


# ============================================================
# 15.2 CARNES: GRAMOS POR PIEZA
# ============================================================

PESO_DIARIO_CARNES = (
    generar_graficos_grupo(
        grupo="CARNES",
        codigos=GRUPOS_PRODUCTOS[
            "CARNES"
        ],
        unidad_medida="gramos por pieza",
        multiplicador=1000
    )
)


# ============================================================
# 15.3 VERDURAS: KG POR PRESENTACIÓN
# ============================================================

PESO_DIARIO_VERDURAS = (
    generar_graficos_grupo(
        grupo="VERDURAS",
        codigos=GRUPOS_PRODUCTOS[
            "VERDURAS"
        ],
        unidad_medida="kg por presentación",
        multiplicador=1
    )
)


# ============================================================
# 15.4 TABLA CONSOLIDADA DE PESO DIARIO
# ============================================================

TABLAS_DIARIAS = [
    tabla
    for tabla in [
        PESO_DIARIO_HELADOS,
        PESO_DIARIO_CARNES,
        PESO_DIARIO_VERDURAS
    ]
    if not tabla.empty
]

if TABLAS_DIARIAS:

    PESO_DIARIO = pd.concat(
        TABLAS_DIARIAS,
        ignore_index=True
    )

else:

    PESO_DIARIO = pd.DataFrame()


print(
    "Gráficos de peso generados:",
    len(RUTAS_GRAFICOS)
)

print(
    "Registros diarios consolidados:",
    len(PESO_DIARIO)
)

display(
    PESO_DIARIO.head(30)
)


# ============================================================
# 15.5 GRÁFICO DE CONSUMO REAL FRENTE AL ESPERADO
# ============================================================

DATOS_CONSUMO_GRAFICO = DETALLE.loc[
    DETALLE["cantidad_esperada"].notna()
].copy()

if not DATOS_CONSUMO_GRAFICO.empty:

    figura_consumo = px.scatter(
        DATOS_CONSUMO_GRAFICO,
        x="cantidad_esperada",
        y="cantidad_real",
        color="tipo_novedad",
        symbol="tipo_componente",
        hover_data=[
            "fecha_produccion",
            "orden",
            "cod_producto",
            "cod_componente",
            "descripcion_componente",
            "cantidad_esperada",
            "cantidad_real",
            "diferencia",
            "porcentaje_desviacion"
        ],
        title=(
            "Consumo real frente al esperado"
        ),
        labels={
            "cantidad_esperada":
                "Cantidad esperada",
            "cantidad_real":
                "Cantidad real",
            "tipo_novedad":
                "Resultado",
            "tipo_componente":
                "Tipo de componente"
        }
    )

    valor_maximo = np.nanmax(
        [
            DATOS_CONSUMO_GRAFICO[
                "cantidad_esperada"
            ].max(),
            DATOS_CONSUMO_GRAFICO[
                "cantidad_real"
            ].max()
        ]
    )

    if np.isfinite(valor_maximo):

        figura_consumo.add_shape(
            type="line",
            x0=0,
            y0=0,
            x1=valor_maximo,
            y1=valor_maximo,
            line=dict(
                color="gray",
                dash="dash"
            )
        )

    figura_consumo.update_layout(
        template="plotly_white",
        height=550
    )

    figura_consumo.show()

    archivo_consumo = (
        "grafico_consumo_real_esperado.html"
    )

    figura_consumo.write_html(
        RUTA_SALIDA / archivo_consumo,
        include_plotlyjs="cdn"
    )

    RUTAS_GRAFICOS[
        "consumo_real_esperado"
    ] = archivo_consumo

else:

    print(
        "No hay datos disponibles para "
        "el gráfico de consumo."
    )


# ============================================================
# 15.6 COMPONENTES NO PARAMETRIZADOS POR FECHA
# ============================================================

if not NO_PARAMETRIZADOS.empty:

    NO_PARAMETRIZADOS_FECHA = (
        NO_PARAMETRIZADOS.groupby(
            [
                "fecha_produccion",
                "cod_producto",
                "cod_componente",
                "descripcion_material"
            ],
            as_index=False,
            dropna=False
        )
        .agg(
            ordenes_afectadas=(
                "orden",
                "nunique"
            ),
            cantidad_real_sap=(
                "cantidad_real_sap",
                "sum"
            )
        )
    )

    figura_no_parametrizados = px.bar(
        NO_PARAMETRIZADOS_FECHA,
        x="fecha_produccion",
        y="ordenes_afectadas",
        color="descripcion_material",
        hover_data=[
            "cod_producto",
            "cod_componente",
            "cantidad_real_sap"
        ],
        title=(
            "Componentes no parametrizados "
            "por fecha"
        ),
        labels={
            "fecha_produccion":
                "Fecha de producción",
            "ordenes_afectadas":
                "Órdenes afectadas",
            "descripcion_material":
                "Componente"
        }
    )

    figura_no_parametrizados.update_layout(
        template="plotly_white",
        height=600,
        xaxis_title="Fecha de producción",
        yaxis_title="Órdenes afectadas"
    )

    figura_no_parametrizados.update_xaxes(
        tickformat="%d/%m/%Y"
    )

    figura_no_parametrizados.show()

    archivo_no_parametrizados = (
        "grafico_componentes_"
        "no_parametrizados.html"
    )

    figura_no_parametrizados.write_html(
        RUTA_SALIDA
        / archivo_no_parametrizados,
        include_plotlyjs="cdn"
    )

    RUTAS_GRAFICOS[
        "componentes_no_parametrizados"
    ] = archivo_no_parametrizados

else:

    NO_PARAMETRIZADOS_FECHA = (
        pd.DataFrame()
    )

    print(
        "No hay componentes no parametrizados."
    )


# ============================================================
# 15.7 RESUMEN FINAL DE GRÁFICOS
# ============================================================

RESUMEN_GRAFICOS = pd.DataFrame(
    [
        {
            "grafico": nombre,
            "archivo": archivo
        }
        for nombre, archivo
        in RUTAS_GRAFICOS.items()
    ]
)

print(
    "Total de gráficos generados:",
    len(RESUMEN_GRAFICOS)
)

display(
    RESUMEN_GRAFICOS
)

Directorio actual: /content
Carpeta de salida: /content/resultados_actividad_5
¿Existe la carpeta?: True
GRÁFICOS DE CONTROL: HELADOS


GRÁFICOS DE CONTROL: CARNES


GRÁFICOS DE CONTROL: VERDURAS


Gráficos de peso generados: 6
Registros diarios consolidados: 25


,cod_producto,descripcion_producto,fecha_produccion,peso_total_real_kg,unidades_producidas,presentaciones_producidas,limite_inferior,limite_superior,ordenes,cantidad_ordenes,valor_control,resultado_peso,grupo_producto
0,14002692,AVALANCHA OREO CJ(63UN),2026-05-22,"2,423.5000","13,104.0000",208.0000,0.1800,0.1950,1082985,1,0.1849,Peso dentro del rango,HELADOS
1,14002692,AVALANCHA OREO CJ(63UN),2026-05-24,933.2600,"5,040.0000",80.0000,0.1800,0.1950,1083164,1,0.1852,Peso dentro del rango,HELADOS
2,14002692,AVALANCHA OREO CJ(63UN),2026-05-25,"2,082.7800","11,340.0000",180.0000,0.1800,0.1950,1083177,1,0.1837,Peso dentro del rango,HELADOS
3,14003025,SUNDAE DE CHOCOLATE CJ(105UN),2026-05-22,"1,583.7200","12,600.0000",120.0000,0.1200,0.1300,1083022,1,0.1257,Peso dentro del rango,HELADOS
4,14002427,BIG CRUNCH MARINADO BOL(10UN),2026-05-18,950.7200,"18,310.0000","1,831.0000",0.0550,0.0600,1082921 | 1082933,2,0.0519,Peso inferior,CARNES
5,14002427,BIG CRUNCH MARINADO BOL(10UN),2026-05-19,318.1000,"5,480.0000",548.0000,0.0550,0.0600,1082963,1,0.0580,Peso dentro del rango,CARNES
6,14002427,BIG CRUNCH MARINADO BOL(10UN),2026-05-20,909.0000,"14,210.0000","1,421.0000",0.0550,0.0600,1083012 | 1083019,2,0.0640,Peso superior,CARNES
7,14002427,BIG CRUNCH MARINADO BOL(10UN),2026-05-21,724.2000,"13,410.0000","1,341.0000",0.0550,0.0600,1083071,1,0.0540,Peso inferior,CARNES
8,14002427,BIG CRUNCH MARINADO BOL(10UN),2026-05-22,"1,584.9000","26,990.0000","2,699.0000",0.0550,0.0600,1083101 | 1083128,2,0.0587,Peso dentro del rango,CARNES
9,14002427,BIG CRUNCH MARINADO BOL(10UN),2026-05-23,"1,628.9000","28,260.0000","2,826.0000",0.0550,0.0600,1083142,1,0.0576,Peso dentro del rango,CARNES


Total de gráficos generados: 8


,grafico,archivo
0,helado_14002692,grafico_helado_14002692.html
1,helado_14003025,grafico_helado_14003025.html
2,carne_14002427,grafico_carne_14002427.html
3,carne_14002441,grafico_carne_14002441.html
4,verdura_12002518,grafico_verdura_12002518.html
5,verdura_14002428,grafico_verdura_14002428.html
6,consumo_real_esperado,grafico_consumo_real_esperado.html
7,componentes_no_parametrizados,grafico_componentes_no_parametrizados.html


## 16. Informe HTML y exportación

In [55]:
# ============================================================
# 16. INFORME ANALITICO EN HTML Y EXPORTACION
# ============================================================
def preparar_tabla_html(tabla, columnas, nombres=None, limite=300):
    """Convierte un DataFrame en una tabla HTML."""
    disponibles = [columna for columna in columnas if columna in tabla.columns]

    if not disponibles:
        return '<div class="alerta">No hay columnas disponibles para mostrar.</div>'

    vista = tabla[disponibles].head(limite).copy()

    if nombres:
        vista = vista.rename(columns={columna: nombres[columna] for columna in disponibles if columna in nombres})

    for columna in vista.columns:
        if pd.api.types.is_datetime64_any_dtype(vista[columna]):
            vista[columna] = vista[columna].dt.strftime("%d/%m/%Y")
        elif pd.api.types.is_numeric_dtype(vista[columna]):
            vista[columna] = vista[columna].round(4)

    return vista.to_html(index=False, border=0, classes="tabla-informe", na_rep="", justify="center")


def crear_tarjeta(titulo, valor, subtitulo, clase):
    """Crea una tarjeta de indicador."""
    return f'''<div class="tarjeta {clase}">
        <div class="tarjeta-titulo">{escape(str(titulo))}</div>
        <div class="tarjeta-valor">{escape(str(valor))}</div>
        <div class="tarjeta-subtitulo">{escape(str(subtitulo))}</div>
    </div>'''


def extraer_cuerpo_html(ruta_archivo):
    """Extrae el contenido visible de un archivo HTML de Plotly."""
    ruta = Path(ruta_archivo)

    if not ruta.exists():
        return f'<div class="alerta">No se encontro el grafico: {escape(ruta.name)}</div>'

    contenido = ruta.read_text(encoding="utf-8")
    inicio = contenido.lower().find("<body>")
    fin = contenido.lower().rfind("</body>")

    if inicio >= 0 and fin > inicio:
        return contenido[inicio + len("<body>"):fin]

    return contenido


def crear_seccion_grafico(titulo, nombre_archivo):
    """Inserta un grafico Plotly dentro del informe principal."""
    ruta_grafico = RUTA_SALIDA / nombre_archivo
    cuerpo_grafico = extraer_cuerpo_html(ruta_grafico)

    return f'''<section class="seccion">
        <h3>{escape(str(titulo))}</h3>
        <div class="grafico-contenedor">{cuerpo_grafico}</div>
    </section>'''


# ============================================================
# 16.1 INDICADORES GENERALES
# ============================================================

TOTAL_ORDENES = RESUMEN["orden"].nunique()
ORDENES_VALIDADAS = RESUMEN.loc[RESUMEN["estado_general"].eq("Validada"), "orden"].nunique()
ORDENES_REPORTAR = RESUMEN.loc[RESUMEN["estado_general"].ne("Validada"), "orden"].nunique()
ORDENES_ADVERTENCIA = RESUMEN.loc[RESUMEN["tiene_advertencia_parametrizacion"].eq("SI"), "orden"].nunique()
CUMPLIMIENTO = ORDENES_VALIDADAS / TOTAL_ORDENES * 100 if TOTAL_ORDENES > 0 else 0
FECHA_MINIMA = RESUMEN["fecha_produccion"].min()
FECHA_MAXIMA = RESUMEN["fecha_produccion"].max()

if pd.notna(FECHA_MINIMA) and pd.notna(FECHA_MAXIMA):
    PERIODO_ANALIZADO = f"{FECHA_MINIMA:%d/%m/%Y} al {FECHA_MAXIMA:%d/%m/%Y}"
else:
    PERIODO_ANALIZADO = "Sin fechas disponibles"

TARJETAS_HTML = f'''<div class="contenedor-tarjetas">
    {crear_tarjeta("Ordenes analizadas", TOTAL_ORDENES, PERIODO_ANALIZADO, "tarjeta-azul")}
    {crear_tarjeta("Ordenes validadas", ORDENES_VALIDADAS, f"{CUMPLIMIENTO:.2f}% de cumplimiento", "tarjeta-verde")}
    {crear_tarjeta("Ordenes para reportar", ORDENES_REPORTAR, "Requieren revision", "tarjeta-roja")}
    {crear_tarjeta("Ordenes con advertencia", ORDENES_ADVERTENCIA, "Componentes sin estandar", "tarjeta-naranja")}
</div>'''


# ============================================================
# 16.2 TABLAS DEL INFORME
# ============================================================

ORDENES_PARA_REPORTAR = RESUMEN.loc[RESUMEN["estado_general"].ne("Validada")].copy()
ORDENES_PARA_REPORTAR = ORDENES_PARA_REPORTAR.sort_values(["fecha_produccion", "cod_producto", "orden"])

TABLA_ORDENES_HTML = preparar_tabla_html(
    ORDENES_PARA_REPORTAR,
    ["fecha_produccion", "orden", "cod_producto", "descripcion_producto", "cantidad_producida_sap", "resultado_peso", "resultado_materias_primas", "resultado_insumos", "estado_general", "recomendacion"],
    {
        "fecha_produccion": "Fecha",
        "orden": "Orden",
        "cod_producto": "Producto",
        "descripcion_producto": "Descripcion",
        "cantidad_producida_sap": "Cantidad producida",
        "resultado_peso": "Resultado peso",
        "resultado_materias_primas": "Materias primas",
        "resultado_insumos": "Insumos",
        "estado_general": "Estado general",
        "recomendacion": "Recomendacion",
    },
)

PESO_INFORME = PESO.copy()
ES_HELADO = PESO_INFORME["cod_producto"].isin(GRUPOS_PRODUCTOS["HELADOS"])
ES_CARNE = PESO_INFORME["cod_producto"].isin(GRUPOS_PRODUCTOS["CARNES"])
ES_VERDURA = PESO_INFORME["cod_producto"].isin(GRUPOS_PRODUCTOS["VERDURAS"])
PESO_INFORME["unidad_informe"] = np.select(
    [ES_HELADO, ES_CARNE, ES_VERDURA],
    ["gramos por helado", "gramos por pieza", "kg por presentacion"],
    default="kg",
)
PESO_INFORME["peso_mostrado"] = PESO_INFORME["valor_control"]
PESO_INFORME["limite_inferior_mostrado"] = PESO_INFORME["limite_inferior"]
PESO_INFORME["limite_superior_mostrado"] = PESO_INFORME["limite_superior"]
CONVERTIR_A_GRAMOS = ES_HELADO | ES_CARNE
PESO_INFORME.loc[CONVERTIR_A_GRAMOS, "peso_mostrado"] *= 1000
PESO_INFORME.loc[CONVERTIR_A_GRAMOS, "limite_inferior_mostrado"] *= 1000
PESO_INFORME.loc[CONVERTIR_A_GRAMOS, "limite_superior_mostrado"] *= 1000

TABLA_PESO_HTML = preparar_tabla_html(
    PESO_INFORME.sort_values(["fecha_produccion", "cod_producto", "orden"]),
    ["fecha_produccion", "orden", "cod_producto", "descripcion_producto", "peso_mostrado", "limite_inferior_mostrado", "limite_superior_mostrado", "unidad_informe", "resultado_peso"],
    {
        "fecha_produccion": "Fecha",
        "orden": "Orden",
        "cod_producto": "Producto",
        "descripcion_producto": "Descripcion",
        "peso_mostrado": "Peso real",
        "limite_inferior_mostrado": "Limite inferior",
        "limite_superior_mostrado": "Limite superior",
        "unidad_informe": "Unidad",
        "resultado_peso": "Resultado",
    },
)

DETALLE_NOVEDADES = DETALLE.loc[DETALLE["tipo_novedad"].ne("Sin novedad")].copy()

if "porcentaje_desviacion" in DETALLE_NOVEDADES.columns:
    DETALLE_NOVEDADES = DETALLE_NOVEDADES.sort_values("porcentaje_desviacion", key=lambda serie: serie.abs(), ascending=False)

TABLA_NOVEDADES_HTML = preparar_tabla_html(
    DETALLE_NOVEDADES,
    ["fecha_produccion", "orden", "cod_producto", "cod_componente", "descripcion_componente", "tipo_componente", "cantidad_esperada", "cantidad_real", "diferencia", "porcentaje_desviacion", "tipo_novedad"],
    {
        "fecha_produccion": "Fecha",
        "orden": "Orden",
        "cod_producto": "Producto",
        "cod_componente": "Componente",
        "descripcion_componente": "Descripcion",
        "tipo_componente": "Tipo",
        "cantidad_esperada": "Esperado",
        "cantidad_real": "Real",
        "diferencia": "Diferencia",
        "porcentaje_desviacion": "Desviacion %",
        "tipo_novedad": "Resultado",
    },
    400,
)

TABLA_NO_PARAMETRIZADOS_HTML = preparar_tabla_html(
    NO_PARAMETRIZADOS.sort_values(["fecha_produccion", "cod_producto", "orden"]),
    ["fecha_produccion", "orden", "cod_producto", "cod_componente", "descripcion_material", "cantidad_real_sap", "estado", "motivo"],
    {
        "fecha_produccion": "Fecha",
        "orden": "Orden",
        "cod_producto": "Producto",
        "cod_componente": "Componente",
        "descripcion_material": "Descripcion",
        "cantidad_real_sap": "Cantidad SAP",
        "estado": "Estado",
        "motivo": "Motivo",
    },
    400,
)

TABLA_PRUEBAS_HTML = preparar_tabla_html(
    PRUEBAS_INTERNAS,
    ["prueba", "resultado", "valor"],
    {"prueba": "Prueba", "resultado": "Resultado", "valor": "Valor obtenido"},
    100,
)


# ============================================================
# 16.3 GRAFICOS INTERACTIVOS EMBEBIDOS
# ============================================================

SECCIONES_GRAFICOS = []

for nombre, archivo in RUTAS_GRAFICOS.items():
    titulo_grafico = nombre.replace("_", " ").title()
    SECCIONES_GRAFICOS.append(crear_seccion_grafico(titulo_grafico, archivo))

GRAFICOS_HTML = "\n".join(SECCIONES_GRAFICOS)


# ============================================================
# 16.4 ESTILOS
# ============================================================

ESTILOS_HTML = '''
<style>
body { margin: 0; padding: 0; background: #f4f6f8; color: #243447; font-family: Arial, Helvetica, sans-serif; }
.contenedor-principal { width: 95%; max-width: 1500px; margin: 20px auto; }
.encabezado { background: linear-gradient(135deg, #174a7e, #2878b5); color: white; padding: 30px; border-radius: 12px; box-shadow: 0 4px 12px rgba(0,0,0,.15); }
.encabezado h1 { margin: 0 0 10px 0; font-size: 30px; }
.encabezado p { margin: 6px 0; line-height: 1.5; }
.contenedor-tarjetas { display: grid; grid-template-columns: repeat(auto-fit, minmax(220px, 1fr)); gap: 15px; margin: 20px 0; }
.tarjeta { color: white; border-radius: 10px; padding: 20px; box-shadow: 0 3px 8px rgba(0,0,0,.15); }
.tarjeta-azul { background: #2878b5; }
.tarjeta-verde { background: #2e8b57; }
.tarjeta-roja { background: #c0392b; }
.tarjeta-naranja { background: #d68910; }
.tarjeta-titulo { font-size: 14px; font-weight: bold; text-transform: uppercase; }
.tarjeta-valor { font-size: 32px; font-weight: bold; margin: 12px 0; }
.tarjeta-subtitulo { font-size: 13px; opacity: .95; }
.seccion { background: white; padding: 25px; margin: 20px 0; border-radius: 12px; box-shadow: 0 3px 10px rgba(0,0,0,.10); overflow-x: auto; }
.seccion h2, .seccion h3 { color: #174a7e; }
.seccion h2 { border-bottom: 2px solid #2878b5; padding-bottom: 8px; }
.tabla-informe { width: 100%; border-collapse: collapse; font-size: 13px; margin-top: 15px; }
.tabla-informe th { background: #174a7e; color: white; padding: 10px; border: 1px solid #d9e2ec; position: sticky; top: 0; }
.tabla-informe td { padding: 8px; border: 1px solid #d9e2ec; text-align: center; vertical-align: top; }
.tabla-informe tr:nth-child(even) { background: #f2f6fa; }
.tabla-informe tr:hover { background: #dbeaf7; }
.grafico-contenedor { width: 100%; min-height: 520px; }
.alerta { padding: 15px; border-left: 6px solid #d68910; background: #fcf3cf; margin: 15px 0; border-radius: 6px; }
.respuesta { padding: 20px; border-left: 6px solid #c0392b; background: #fadbd8; border-radius: 6px; font-size: 16px; }
.pie { background: #174a7e; color: white; padding: 20px; border-radius: 10px; margin-top: 25px; text-align: center; font-size: 12px; }
</style>
'''


# ============================================================
# 16.5 CONSTRUIR Y GUARDAR INFORME
# ============================================================

INFORME_HTML = f'''<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Informe analitico de produccion</title>
    {ESTILOS_HTML}
</head>
<body>
<div class="contenedor-principal">
    <header class="encabezado">
        <h1>Informe analitico de ordenes de produccion</h1>
        <p><strong>Autores:</strong> Laila Tatiana Cardenas Guerrero, Luis Eduardo Ortega Montes</p>
        <p><strong>Periodo detectado:</strong> {escape(PERIODO_ANALIZADO)}</p>
        <p><strong>Pregunta:</strong> ¿Que ordenes deben reportarse por no cumplir los criterios de cierre y liberacion?</p>
    </header>

    {TARJETAS_HTML}

    <section class="seccion">
        <h2>1. Resultado general</h2>
        <div class="respuesta">De las <strong>{TOTAL_ORDENES}</strong> ordenes analizadas, <strong>{ORDENES_REPORTAR}</strong> deben ser revisadas por Produccion y Costos.</div>
    </section>

    <section class="seccion">
        <h2>2. Ordenes para reportar</h2>
        {TABLA_ORDENES_HTML}
    </section>

    <section class="seccion">
        <h2>3. Control de peso</h2>
        <p>Helados en gramos por helado, carnes en gramos por pieza y verduras en kilogramos por presentacion.</p>
        {TABLA_PESO_HTML}
    </section>

    <section class="seccion">
        <h2>4. Novedades de componentes</h2>
        {TABLA_NOVEDADES_HTML}
    </section>

    <section class="seccion">
        <h2>5. Advertencias de parametrizacion</h2>
        <div class="alerta">Se encontraron <strong>{len(NO_PARAMETRIZADOS)}</strong> combinaciones orden-componente sin estandar. Estas advertencias no cambian el estado general.</div>
        {TABLA_NO_PARAMETRIZADOS_HTML}
    </section>

    <section class="seccion">
        <h2>6. Graficos interactivos</h2>
        <p>Los graficos muestran el comportamiento por producto y fecha.</p>
    </section>

    {GRAFICOS_HTML}

    <section class="seccion">
        <h2>7. Pruebas internas</h2>
        {TABLA_PRUEBAS_HTML}
    </section>

    <section class="seccion">
        <h2>8. Recomendaciones</h2>
        <ol>
            <li>Revisar primero las ordenes con peso fuera de limite.</li>
            <li>Mantener la validacion exacta de los insumos.</li>
            <li>Mostrar siempre la diferencia entre consumo esperado y real.</li>
            <li>Parametrizar progresivamente los componentes advertidos.</li>
            <li>Analizar la evolucion por fecha y producto.</li>
        </ol>
    </section>

    <footer class="pie">Informe generado automaticamente desde Google Colab.<br>Actividad 5: Analitica de ordenes de produccion.</footer>
</div>
</body>
</html>'''

RUTA_INFORME_HTML = RUTA_SALIDA / "INFORME_ANALITICO.html"
RUTA_INFORME_HTML.write_text(INFORME_HTML, encoding="utf-8")
print("Informe HTML creado en:", RUTA_INFORME_HTML)


# ============================================================
# 16.6 EXPORTAR CSV Y CREAR ZIP
# ============================================================

SALIDAS = {
    "RESUMEN_POR_ORDEN": RESUMEN,
    "DETALLE_COMPLETO": DETALLE,
    "DETALLE_NOVEDADES": DETALLE_NOVEDADES,
    "CONTROL_PESO": PESO_INFORME,
    "PESO_DIARIO": PESO_DIARIO,
    "COMPONENTES_NO_PARAMETRIZADOS": NO_PARAMETRIZADOS,
    "INDICADORES": INDICADORES,
    "PRUEBAS_INTERNAS": PRUEBAS_INTERNAS,
    "DIMENSIONES_ORIGINALES": DIMENSIONES_ORIGINALES,
    "DIMENSIONES_NORMALIZADAS": DIMENSIONES_NORMALIZADAS,
}

for nombre, tabla in SALIDAS.items():
    tabla.to_csv(RUTA_SALIDA / f"{nombre}.csv", sep=";", index=False, encoding="utf-8-sig")

RUTA_ZIP = shutil.make_archive("resultados_actividad_5", "zip", RUTA_SALIDA)
print("Paquete ZIP creado en:", RUTA_ZIP)


# ============================================================
# 16.7 VISTA PREVIA EN COLAB
# ============================================================

display(HTML('<div style="padding:15px;background:#eaf2f8;border-left:6px solid #2878b5;border-radius:6px;margin:15px 0;"><strong>Informe HTML generado correctamente.</strong><br>La vista previa aparece debajo.</div>'))
display(HTML(INFORME_HTML))

Informe HTML creado en: resultados_actividad_5/INFORME_ANALITICO.html
Paquete ZIP creado en: /content/resultados_actividad_5.zip


Fecha,Orden,Producto,Descripcion,Cantidad producida,Resultado peso,Materias primas,Insumos,Estado general,Recomendacion
18/05/2026,1082919,14002441,STRIPS MARINADO BOL(20UN),375.0000,Peso dentro del rango,Reportar por consumo superior,Reportar por consumo superior,Reportar por consumo superior,Reportar a Producción y Costos
18/05/2026,1082931,14002441,STRIPS MARINADO BOL(20UN),247.0000,Peso inferior,Reportar por consumo superior,Reportar por consumo superior,Reportar por consumo superior | Reportar por peso inferior,Reportar a Producción y Costos
19/05/2026,1082952,12002518,MIX VERDURAS ENSALADA KFC,964.2000,Peso dentro del rango,Reportar por consumo inferior | Reportar por merma superior,Reportar por consumo inferior,Reportar por consumo inferior | Reportar por merma superior,Reportar a Producción y Costos
19/05/2026,1082994,12002518,MIX VERDURAS ENSALADA KFC,771.5000,Peso dentro del rango,Reportar por consumo inferior | Reportar por merma superior,Reportar por consumo inferior,Reportar por consumo inferior | Reportar por merma superior,Reportar a Producción y Costos
19/05/2026,1082941,14002428,LECHUGA BATAVIA PROCESADA BOL(1KG),337.0000,Peso dentro del rango,Reportar por merma superior,Reportar por consumo superior,Reportar por consumo superior | Reportar por merma superior,Reportar a Producción y Costos
19/05/2026,1082965,14002441,STRIPS MARINADO BOL(20UN),405.0000,Peso inferior,Reportar por consumo superior,Reportar por consumo superior,Reportar por consumo superior | Reportar por peso inferior,Reportar a Producción y Costos
19/05/2026,1082986,14002441,STRIPS MARINADO BOL(20UN),100.0000,Peso superior,Reportar por consumo superior,Reportar por consumo superior,Reportar por consumo superior | Reportar por peso superior,Reportar a Producción y Costos
20/05/2026,1082991,14002428,LECHUGA BATAVIA PROCESADA BOL(1KG),344.0000,Peso dentro del rango,Reportar por merma superior,Reportar por consumo superior,Reportar por consumo superior | Reportar por merma superior,Reportar a Producción y Costos
20/05/2026,1083018,14002441,STRIPS MARINADO BOL(20UN),623.0000,Peso dentro del rango,Reportar por consumo superior,Reportar por consumo superior,Reportar por consumo superior,Reportar a Producción y Costos
20/05/2026,1083020,14002441,STRIPS MARINADO BOL(20UN),150.0000,Peso dentro del rango,Reportar por consumo superior,Reportar por consumo superior,Reportar por consumo superior,Reportar a Producción y Costos
